# Computer Vision Using Deep Learning - Practical 10

## YOLO (You Only Look Once)

### 1. Imports

In [19]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image
import numpy as np
import os
import matplotlib.pyplot as plt
import torch.optim as optim
from tqdm import tqdm
import cv2

### 2. YOLO Model Definition

In [20]:
class YOLO(nn.Module):
    def __init__(self, S=7, B=4, C=1):
        super(YOLO, self).__init__()
        self.S = S
        self.B = B
        self.C = C
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, 1, 1), nn.LeakyReLU(0.1),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, 1, 1), nn.LeakyReLU(0.1),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.LeakyReLU(0.1),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, 1, 1), nn.LeakyReLU(0.1),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(256, 512, 3, 1, 1), nn.LeakyReLU(0.1),
            nn.AdaptiveAvgPool2d((S, S))
        )
        self.fcs = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * S * S, 1024),
            nn.LeakyReLU(0.1),
            nn.Linear(1024, S * S * (C + B * 5))
        )

    def forward(self, x):
        x = self.features(x)
        x = self.fcs(x)
        x = x.view(-1, self.S, self.S, self.C + self.B * 5)
        x[..., :self.B * 5] = torch.sigmoid(x[..., :self.B * 5]) 
        return x

### 3. Bounding Box IoU (Intersection over Union)

In [21]:
def bbox_iou(box1, box2):
    box1_x1 = box1[0] - box1[2]/2
    box1_y1 = box1[1] - box1[3]/2
    box1_x2 = box1[0] + box1[2]/2
    box1_y2 = box1[1] + box1[3]/2
    box2_x1 = box2[0] - box2[2]/2
    box2_y1 = box2[1] - box2[3]/2
    box2_x2 = box2[0] + box2[2]/2
    box2_y2 = box2[1] + box2[3]/2

    inter_x1 = max(box1_x1, box2_x1)
    inter_y1 = max(box1_y1, box2_y1)
    inter_x2 = min(box1_x2, box2_x2)
    inter_y2 = min(box1_y2, box2_y2)

    inter_area = max(inter_x2 - inter_x1, 0) * max(inter_y2 - inter_y1, 0)
    box1_area = (box1_x2 - box1_x1) * (box1_y2 - box1_y1)
    box2_area = (box2_x2 - box2_x1) * (box2_y2 - box2_y1)
    union = box1_area + box2_area - inter_area

    if union == 0: return 0
    return inter_area / union

### 4. YOLO Loss Function

In [22]:
class YoloLoss(nn.Module):
    def __init__(self, S=7, B=4, C=1, lambda_coord=5, lambda_noobj=0.5):
        super(YoloLoss, self).__init__()
        self.S = S
        self.B = B
        self.C = C
        self.lambda_coord = lambda_coord
        self.lambda_noobj = lambda_noobj

    def forward(self, predictions, target):
        coord_mask = target[..., 4::5].sum(-1) > 0
        noobj_mask = ~coord_mask

        loss_xy, loss_wh, loss_conf_obj, loss_conf_noobj = 0, 0, 0, 0

        for b in range(self.B):
            pbox = predictions[..., b*5:(b+1)*5]
            tbox = target[..., b*5:(b+1)*5]
            mask = tbox[..., 4] > 0

            loss_xy += ((pbox[mask][..., :2] - tbox[mask][..., :2]) ** 2).sum()
            loss_wh += ((pbox[mask][..., 2:4].sqrt() - tbox[mask][..., 2:4].sqrt()) ** 2).sum()
            loss_conf_obj += ((pbox[mask][..., 4] - tbox[mask][..., 4]) ** 2).sum()
            loss_conf_noobj += ((pbox[~mask][..., 4] - tbox[~mask][..., 4]) ** 2).sum()

        loss_class = ((predictions[coord_mask][..., -1] - target[coord_mask][..., -1]) ** 2).sum()

        total_loss = (
            self.lambda_coord * (loss_xy + loss_wh)
            + loss_conf_obj
            + self.lambda_noobj * loss_conf_noobj
            + loss_class
        )
        return total_loss

### 5. Custom Dataset (PennFudanPed)

In [23]:
class PennFudanDataset(Dataset):
    def __init__(self, root, S=7, B=4, transforms=None, limit=20):
        self.root = root
        self.S = S
        self.B = B
        self.transforms = transforms
        self.imgs = list(sorted(os.listdir(os.path.join(root, "PNGImages"))))[:limit]
        self.masks = list(sorted(os.listdir(os.path.join(root, "PedMasks"))))[:limit]

    def __getitem__(self, idx):
        img_path = os.path.join(self.root, "PNGImages", self.imgs[idx])
        mask_path = os.path.join(self.root, "PedMasks", self.masks[idx])
        img = Image.open(img_path).convert("RGB")
        mask = np.array(Image.open(mask_path))
        H, W = mask.shape
        obj_ids = np.unique(mask)[1:]
        boxes = []

        for obj_id in obj_ids[:self.B]:
            pos = np.where(mask == obj_id)
            xmin, xmax = np.min(pos[1]), np.max(pos[1])
            ymin, ymax = np.min(pos[0]), np.max(pos[0])
            boxes.append([xmin, ymin, xmax, ymax])

        target = torch.zeros((self.S, self.S, self.B * 5 + 1))
        for box in boxes:
            x_center = (box[0] + box[2]) / 2 / W
            y_center = (box[1] + box[3]) / 2 / H
            w = (box[2] - box[0]) / W
            h = (box[3] - box[1]) / H
            i = int(x_center * self.S)
            j = int(y_center * self.S)

            for b in range(self.B):
                if target[j, i, b * 5 + 4] == 0:
                    target[j, i, b * 5:(b + 1) * 5] = torch.tensor([x_center, y_center, w, h, 1.0])
                    break
            target[j, i, -1] = 1.0 

        if self.transforms:
            img = self.transforms(img)

        return img, target

    def __len__(self):
        return len(self.imgs)

### 6. Training Setup & Loop

**Note:** You will need to download and extract the [Penn-Fudan Database for Pedestrian Detection and Segmentation](https://www.cis.upenn.edu/~jshi/ped_html/) and place it in `/content/PennFudanPed` for this to run.

In [ ]:
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])

# !wget https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip -P /content/
# !unzip /content/PennFudanPed.zip -d /content/

root = "/PennFudanPed"
dataset = PennFudanDataset(root, S=7, B=4, transforms=transform, limit=20)
train_loader = DataLoader(dataset, batch_size=4, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = YOLO(S=7, B=4, C=1).to(device)
criterion = YoloLoss(S=7, B=4, C=1)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(6):
    model.train()
    total_loss = 0
    for imgs, targets in tqdm(train_loader):
        imgs, targets = imgs.to(device), targets.to(device)
        preds = model(imgs)
        loss = criterion(preds, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss / len(train_loader):.4f}")

FileNotFoundError: [WinError 3] The system cannot find the path specified: '/PennFudanPed\\PNGImages'

### 7. Visualize Predictions

In [ ]:
def visualize_predictions(model, dataset, num_images=5, conf_threshold=0.5):
    model.eval()
    fig, axes = plt.subplots(1, num_images, figsize=(20, 8))
    
    for i in range(num_images):
        img, _ = dataset[i]
        with torch.no_grad():
            preds = model(img.unsqueeze(0).to(device)).cpu().squeeze(0)

        img_np = np.array(T.ToPILImage()(img))
        H, W, _ = img_np.shape
        S = preds.shape[0]
        cell_size_x = W / S
        cell_size_y = H / S

        for y in range(S):
            for x in range(S):
                for b in range(4):
                    box = preds[y, x, b * 5:(b + 1) * 5]
                    conf = box[4].item()

                    if conf > conf_threshold:
                        bx = (box[0].item() * W)
                        by = (box[1].item() * H)
                        bw = (box[2].item() * W)
                        bh = (box[3].item() * H)

                        xmin = int(bx - bw / 2)
                        ymin = int(by - bh / 2)
                        xmax = int(bx + bw / 2)
                        ymax = int(by + bh / 2)

                        # Use a copy to avoid UserWarning about non-writable array
                        img_np = cv2.rectangle(img_np.copy(), (xmin, ymin), (xmax, ymax), (255, 0, 0), 2)

        axes[i].imshow(img_np)
        axes[i].axis("off")
        
    plt.show()

In [ ]:
visualize_predictions(model, dataset, num_images=5, conf_threshold=0.5)